## Custom Layers

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

### Layers without parameters

In [2]:
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [3]:
layer = CenteredLayer()
layer(torch.tensor([1.0, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [4]:
test_x = torch.tensor([1.0, 2, 3, 4, 5])
test_x, test_x.mean(), test_x - test_x.mean()

(tensor([1., 2., 3., 4., 5.]), tensor(3.), tensor([-2., -1.,  0.,  1.,  2.]))

In [5]:
net = nn.Sequential(nn.LazyLinear(128), CenteredLayer())

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [9]:
Y = net(torch.rand(4, 8))
Y.mean(), Y

(tensor(5.5879e-09, grad_fn=<MeanBackward0>),
 tensor([[-7.9883e-01,  2.9786e-02,  3.4278e-01, -1.6015e-01,  1.6785e-01,
           8.9051e-02, -1.2566e-01,  2.5082e-01,  8.1833e-02,  2.4672e-01,
           4.5420e-02, -2.6874e-01,  3.6652e-01,  3.7211e-01, -1.3870e-01,
           4.5152e-01, -3.4298e-02, -4.5822e-02, -2.7880e-01, -1.0003e-01,
          -6.6814e-02,  9.6070e-02,  3.2580e-01, -2.4943e-01,  5.8226e-02,
          -1.9695e-01, -6.6675e-01, -6.9737e-01, -1.1821e-01, -2.4760e-01,
          -1.0317e-01,  3.1833e-01,  5.0375e-01, -3.8568e-01, -3.3010e-01,
           3.5911e-01,  6.0760e-02, -4.4922e-01, -3.2084e-01, -8.8638e-02,
           1.1050e-02,  5.4553e-01, -9.5397e-01, -3.3318e-01, -3.4082e-01,
           1.4312e-01, -1.2903e-01, -4.8223e-01,  7.7804e-02,  3.6976e-01,
          -2.7760e-03,  7.4456e-01,  6.0881e-02, -2.5534e-02, -3.0406e-01,
          -2.1467e-01,  1.8450e-01, -1.8894e-01, -6.6286e-02,  5.6965e-01,
           4.5414e-01,  4.2886e-01,  2.0915e-01, -4.13

### Layers with parameters

In [10]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

In [11]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[-1.4762, -1.0920, -0.1107],
        [-0.4123, -0.6671,  0.0613],
        [ 1.4291,  0.3894, -0.9377],
        [ 1.1635, -0.2758,  1.2211],
        [-0.4038, -0.4540,  1.5520]], requires_grad=True)

In [12]:
linear(torch.rand(2, 5))

tensor([[0.8503, 0.0787, 0.8282],
        [0.0000, 0.0000, 0.0776]])

In [13]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]])

## Exercises

In [25]:
class MyReduced(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features, in_features))

    def forward(self, X):
        y = torch.einsum('kij,bi,bj->bk', self.weight, X, X)
        return F.relu(y)

In [21]:
test_X = torch.rand(1, 5)
test_X, torch.matmul(test_X.T, test_X)

(tensor([[0.7498, 0.7621, 0.7822, 0.1833, 0.4717]]),
 tensor([[0.5622, 0.5714, 0.5864, 0.1375, 0.3536],
         [0.5714, 0.5808, 0.5961, 0.1397, 0.3594],
         [0.5864, 0.5961, 0.6118, 0.1434, 0.3689],
         [0.1375, 0.1397, 0.1434, 0.0336, 0.0865],
         [0.3536, 0.3594, 0.3689, 0.0865, 0.2225]]))

In [22]:
test_X.T, test_X

(tensor([[0.7498],
         [0.7621],
         [0.7822],
         [0.1833],
         [0.4717]]),
 tensor([[0.7498, 0.7621, 0.7822, 0.1833, 0.4717]]))